In [1]:

import pandas as pd
df = pd.read_csv("../data/clean.csv")
print("Размер датасета:", df.shape)
print("Пропущенные значения:", df.isna().sum().sum())
print("Дубликаты:", df.duplicated().sum())
print("\nТипы данных:")
print(df.dtypes.value_counts())
print("\nПроверка оценок:")
for col in ["G1", "G2", "G3"]:
    print(
        f"{col}: min={df[col].min()}, "
        f"max={df[col].max()}"
    )
display(df.head())


Размер датасета: (649, 33)
Пропущенные значения: 0
Дубликаты: 0

Типы данных:
str      17
int64    16
Name: count, dtype: int64

Проверка оценок:
G1: min=0, max=19
G2: min=0, max=19
G3: min=0, max=19


,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,4,3,4,1,1,3,4,0,11,11
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,5,3,3,1,1,3,2,9,11,11
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,4,3,2,2,3,3,6,12,13,12
3,GP,F,15,U,GT3,T,4,2,health,services,...,3,2,2,1,1,5,0,14,14,14
4,GP,F,16,U,GT3,T,3,3,other,other,...,4,3,2,1,2,5,0,11,13,13


In [2]:
raw = pd.read_csv("../data/student-por.csv", sep=";")
clean = pd.read_csv("../data/clean.csv")

comparison = pd.DataFrame({
    "Показатель": [
        "Количество строк",
        "Количество столбцов",
        "Пропущенные значения",
        "Полные дубликаты"
    ],
    "Исходные данные": [
        len(raw),
        raw.shape[1],
        raw.isna().sum().sum(),
        raw.duplicated().sum()
    ],
    "Очищенные данные": [
        len(clean),
        clean.shape[1],
        clean.isna().sum().sum(),
        clean.duplicated().sum()
    ]
})

display(comparison)


,Показатель,Исходные данные,Очищенные данные
0,Количество строк,649,649
1,Количество столбцов,33,33
2,Пропущенные значения,0,0
3,Полные дубликаты,0,0


In [3]:

import pandas as pd

df = pd.read_csv("../data/student-por.csv", sep=";")

cat_cols = df.select_dtypes(
    include=["object", "string"]
).columns

for col in cat_cols:
    print(f"\n{col}: {df[col].nunique()} категорий")
    print(sorted(df[col].dropna().unique().tolist()))



school: 2 категорий
['GP', 'MS']

sex: 2 категорий
['F', 'M']

address: 2 категорий
['R', 'U']

famsize: 2 категорий
['GT3', 'LE3']

Pstatus: 2 категорий
['A', 'T']

Mjob: 5 категорий
['at_home', 'health', 'other', 'services', 'teacher']

Fjob: 5 категорий
['at_home', 'health', 'other', 'services', 'teacher']

reason: 4 категорий
['course', 'home', 'other', 'reputation']

guardian: 3 категорий
['father', 'mother', 'other']

schoolsup: 2 категорий
['no', 'yes']

famsup: 2 категорий
['no', 'yes']

paid: 2 категорий
['no', 'yes']

activities: 2 категорий
['no', 'yes']

nursery: 2 категорий
['no', 'yes']

higher: 2 категорий
['no', 'yes']

internet: 2 категорий
['no', 'yes']

romantic: 2 категорий
['no', 'yes']


In [4]:

results = []

for col in cat_cols:
    s = df[col].astype("string")

    results.append({
        "Признак": col,
        "Категорий": df[col].nunique(),
        "Лишние пробелы": (s != s.str.strip()).sum(),
        "Пустые строки": s.str.strip().eq("").sum()
    })

category_check = pd.DataFrame(results)

display(category_check)

print("Всего лишних пробелов:",
      category_check["Лишние пробелы"].sum())

print("Всего пустых строк:",
      category_check["Пустые строки"].sum())


,Признак,Категорий,Лишние пробелы,Пустые строки
0,school,2,0,0
1,sex,2,0,0
2,address,2,0,0
3,famsize,2,0,0
4,Pstatus,2,0,0
5,Mjob,5,0,0
6,Fjob,5,0,0
7,reason,4,0,0
8,guardian,3,0,0
9,schoolsup,2,0,0


Всего лишних пробелов: 0
Всего пустых строк: 0


In [6]:

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, root_mean_squared_error

df = pd.read_csv("../data/clean.csv")
X = df.drop(columns=["G3"])
y = df["G3"]
num_cols = X.select_dtypes(include="number").columns.tolist()
cat_cols = X.select_dtypes(exclude="number").columns.tolist()
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42
)
num = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler()
)
cat = make_pipeline(
    SimpleImputer(strategy="most_frequent"),
    OneHotEncoder(handle_unknown="ignore")
)
prep = make_column_transformer(
    (num, num_cols),
    (cat, cat_cols)
)
model = make_pipeline(prep, LinearRegression())

cv_r2 = cross_val_score(
    model, X_train, y_train, cv=5, scoring="r2"
)

cv_rmse = -cross_val_score(
    model, X_train, y_train,
    cv=5,
    scoring="neg_root_mean_squared_error"
)
model.fit(X_train, y_train)
pred = model.predict(X_test)
print("Train:", len(X_train))
print("Test:", len(X_test))
print("CV R²:", cv_r2.round(4))
print("Средний CV R²:", round(cv_r2.mean(), 4))
print("Средний CV RMSE:", round(cv_rmse.mean(), 4))
print("Test R²:", round(r2_score(y_test, pred), 4))
print("Test RMSE:", round(root_mean_squared_error(y_test, pred), 4))


Train: 486
Test: 163
CV R²: [0.7825 0.7728 0.8233 0.8665 0.8631]
Средний CV R²: 0.8216
Средний CV RMSE: 1.3625
Test R²: 0.8653
Test RMSE: 1.1655
